# pySCENIC Single-Run GRN/Regulon Inference

This notebook reproduces one run of pySCENIC GRNBoost2 (or GENIE3) inference,
optionally followed by motif-based context pruning into regulons.

## Expected expression matrix format

pySCENIC expects rows to be cells and columns to be genes. If your matrix is
genes x cells, set `TRANSPOSE = True`. CSV and tab-separated text files are
supported; the prepared output matrix is always tab-separated.

The GEO input deposited for this project is also supported directly. Provide
the common filename prefix for one 10x/Matrix Market sample with
`TENX_PREFIX`.

## Configuration

In [17]:
import gzip
import pandas as pd
from scipy.io import mmread
from pathlib import Path

In [12]:
matrix_path = "./data/filtered_counts_300_hvg/matrix.mtx.gz"
genes_path = "./data/filtered_counts_300_hvg/genes.tsv.gz"
barcodes_path = "./data/filtered_counts_300_hvg/barcodes.tsv.gz"

# Read gene names
gene_names = []
with gzip.open(genes_path, "rt") as handle:
    gene_names = [line.rstrip("\n").split("\t")[0] for line in handle if line.strip()]

# Read barcodes
barcodes = []
with gzip.open(barcodes_path, "rt") as handle:
    barcodes = [line.rstrip("\n").split("\t")[0] for line in handle if line.strip()]

matrix = mmread(matrix_path).tocsr()
if matrix.shape != (len(gene_names), len(barcodes)):
    raise ValueError(
        "10x matrix dimensions do not match genes/barcodes: "
        f"matrix={matrix.shape}, genes={len(gene_names)}, barcodes={len(barcodes)}"
    )

dense_cells_by_genes = matrix.transpose().toarray()
dataset = pd.DataFrame(dense_cells_by_genes, index=barcodes, columns=gene_names)
dataset

,Khdc1a,Icos,Serpine2,Btg2,Rgs2,Rgs1,Xcl1,Fcgr3,Fcer1g,Slamf7,...,Ms4a4b,Ms4a2,Ms4a3,Mpeg1,Ifit3,Ifit1bl1,Ifit3b,Ifit1,Dntt,mt-Co1
young_A_AAACCTGAGTTCGATC-1,0.0,0.0,0.000000,1.609118,0.000000,0.000000,0.0,0.0,1.098346,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,4.726992
young_A_AAACCTGCAATCGGTT-1,0.0,0.0,1.139128,0.000000,0.000000,0.000000,0.0,0.0,1.139128,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,4.349834
young_A_AAACCTGCACACGCTG-1,0.0,0.0,0.994147,1.809477,0.994147,0.000000,0.0,0.0,0.994147,0.0,...,0.994147,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,2.792499,3.849408
young_A_AAACCTGCACGGATAG-1,0.0,0.0,0.000000,0.000000,0.000000,0.731263,0.0,0.0,1.149117,0.0,...,0.000000,0.0,1.854501,0.0,0.0,0.0,0.0,0.00000,0.000000,4.116617
young_A_AAACCTGCATCATCCC-1,0.0,0.0,0.000000,0.000000,0.647190,1.534858,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,3.950654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
old_B_TTTGTCAGTCTTGTCC-1,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,1.797617,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,4.233756
old_B_TTTGTCAGTGGTAACG-1,0.0,0.0,0.000000,0.000000,0.789200,1.224736,0.0,0.0,1.224736,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,3.790055
old_B_TTTGTCATCAACCAAC-1,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,3.901829
old_B_TTTGTCATCACAGGCC-1,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.00000,0.000000,4.281595


In [13]:
# We have to save the matrix as a `.tsv` file for pySCENIC:
dataset.to_csv("./data/filtered_counts.tsv", sep="\t")

In [ ]:
# At this point, we could use the TF list that we downloaded in the previous notebook.
# However, at this point, we can also show how such a list can be constructed.

# First, we obtain motif annotations prepared by the pySCENIC team:
!wget "https://resources.aertslab.org/cistarget/motif2tf/motifs-v9-nr.mgi-m0.001-o0.0.tbl"
!mv "motifs-v9-nr.mgi-m0.001-o0.0.tbl" ./data/motif_annotations.tbl

# We will also need a motif ranking database later, also provided by pySCENIC:
!wget "https://resources.aertslab.org/cistarget/databases/mus_musculus/mm9/refseq_r45/mc9nr/gene_based/mm9-tss-centered-10kb-7species.mc9nr.genes_vs_motifs.rankings.feather"
!mv "mm9-tss-centered-10kb-7species.mc9nr.genes_vs_motifs.rankings.feather" ./data/ranking_database.mc9nr.genes_vs_motifs.rankings.feather

--2026-07-12 15:16:44--  https://resources.aertslab.org/cistarget/motif2tf/motifs-v9-nr.mgi-m0.001-o0.0.tbl
Resolving resources.aertslab.org (resources.aertslab.org)... 134.58.50.9
Connecting to resources.aertslab.org (resources.aertslab.org)|134.58.50.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 112121859 (107M)
Saving to: ‘motifs-v9-nr.mgi-m0.001-o0.0.tbl’

motifs-v9-nr.mgi-m0 100%[===================>] 106.93M  77.8MB/s    in 1.4s    

2026-07-12 15:16:45 (77.8 MB/s) - ‘motifs-v9-nr.mgi-m0.001-o0.0.tbl’ saved [112121859/112121859]

--2026-07-12 15:16:46--  https://resources.aertslab.org/cistarget/databases/mus_musculus/mm9/refseq_r45/mc9nr/gene_based/mm9-tss-centered-10kb-7species.mc9nr.genes_vs_motifs.rankings.feather
Resolving resources.aertslab.org (resources.aertslab.org)... 134.58.50.9
Connecting to resources.aertslab.org (resources.aertslab.org)|134.58.50.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 994627274 (949M)

In [19]:
motifs = pd.read_csv("./data/motif_annotations.tbl", sep="\t", low_memory=False)

tf_names = (
    motifs["gene_name"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .tolist()
)

# Write all TFs that appear in the motifs file, one TF per line:
Path("./data/derived_TFs.txt").write_text("\n".join(tf_names) + "\n")

print(f"Loaded {len(tf_names)} TFs")

Loaded 1721 TFs


In [22]:
# Now, we can essentially run what we already know: Grnboost with available TFs on the given dataset:
!pyscenic grn \
    --method grnboost2 \
    --seed 0 \
    --num_workers 4 \
    -o "./data/initial_GRN.tsv" \
    "./data/filtered_counts.tsv" \
    "./data/derived_TFs.txt"


2026-07-12 15:34:08,736 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-07-12 15:34:09,103 - pyscenic.cli.pyscenic - INFO - Inferring regulatory networks.
preparing dask client
parsing input
creating dask graph
4 partitions
computing dask graph
/home/xpastva/bioinformatics-summer-school/venv-pyscenic/lib/python3.10/site-packages/distributed/client.py:3169: UserWarning: Sending large graph of size 33.64 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
not shutting down client, client was created externally
finished

2026-07-12 15:34:33,832 - pyscenic.cli.pyscenic - INFO - Writing results to file.


In [26]:
# Now comes the actual fun part: We let pySCENIC prune the inferred regulations based on 
# what is actually feasible in terms of the regulatory motifs:

!pyscenic ctx \
    -a \
    --annotations_fname "./data/motif_annotations.tbl" \
    --expression_mtx_fname "./data/filtered_counts.tsv" \
    -o "./data/regulon_GRN.tsv" \
    "./data/initial_GRN.tsv" \
    "./data/ranking_database.mc9nr.genes_vs_motifs.rankings.feather"


2026-07-12 15:38:11,418 - pyscenic.cli.pyscenic - INFO - Creating modules.

2026-07-12 15:38:11,428 - pyscenic.cli.pyscenic - INFO - Loading expression matrix.

2026-07-12 15:38:11,918 - pyscenic.utils - INFO - Calculating Pearson correlations.

2026-07-12 15:38:11,919 - pyscenic.utils - WARNING - Note on correlation calculation: the default behaviour for calculating the correlations has changed after pySCENIC verion 0.9.16. Previously, the default was to calculate the correlation between a TF and target gene using only cells with non-zero expression values (mask_dropouts=True). The current default is now to use all cells to match the behavior of the R verision of SCENIC. The original settings can be retained by setting 'rho_mask_dropouts=True' in the modules_from_adjacencies function, or '--mask_dropouts' from the CLI.
	Dropout masking is currently set to [False].

2026-07-12 15:38:12,104 - pyscenic.utils - INFO - Creating modules.
/home/xpastva/bioinformatics-summer-school/venv-pysc

In [ ]:
import networkx as nx
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
regulon_path = "./data/regulon_GRN.tsv"

col_names = [
    "TF", "MotifID", "AUC", "NES", "MotifSimilarityQvalue",
    "OrthologousIdentity", "Annotation", "Context", "TargetGenes", "RankAtMax"
]
regulon_df = pd.read_csv(regulon_path, sep="\t", skiprows=3, names=col_names, low_memory=False)

# Parse TargetGenes strings into lists of (gene, importance) tuples
def parse_target_genes(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

edges = []
for _, row in regulon_df.iterrows():
    targets = parse_target_genes(row["TargetGenes"])
    for gene, importance in targets:
        edges.append({"TF": row["TF"], "Target": gene, "Importance": importance})

edge_df = pd.DataFrame(edges)

# Aggregate duplicate edges by taking the max importance
edge_df = edge_df.groupby(["TF", "Target"], as_index=False)["Importance"].max()
edge_df = edge_df.sort_values("Importance", ascending=False).reset_index(drop=True)

print(f"Regulon GRN: {len(edge_df)} unique TF-target edges ({len(edge_df['TF'].unique())} TFs, {len(edge_df['Target'].unique())} targets)")

In [ ]:
# Cumulative importance analysis & edge distribution
edge_df = edge_df.sort_values(by="Importance", ascending=False).reset_index(drop=True)
edge_df["Rank"] = edge_df.index + 1

total_imp = edge_df["Importance"].sum()
edge_df["Cumulative_Importance_Pct"] = (edge_df["Importance"].cumsum() / total_imp) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=edge_df, x="Importance", bins=50, ax=axes[0], log_scale=(False, True), color="skyblue")
axes[0].set_title("Distribution of Regulon Edge Weights")
axes[0].set_xlabel("Importance")
axes[0].set_ylabel("Count (log)")

axes[1].plot(edge_df["Rank"], edge_df["Cumulative_Importance_Pct"], color="salmon", lw=2)
axes[1].axvline(x=1000, color="gray", linestyle="--", label="Top 1000 Edges")
axes[1].set_title("Cumulative Importance vs. Edge Rank")
axes[1].set_xlabel("Edge Rank")
axes[1].set_ylabel("Cumulative Importance (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Threshold to top 1000 edges for network analysis
thresh = edge_df.head(1000).copy()
print(f"Filtered network: {len(thresh)} edges")

G = nx.from_pandas_edgelist(
    thresh, source="TF", target="Target", edge_attr="Importance", create_using=nx.DiGraph()
)
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Out-degree (master regulators) and in-degree
out_deg = dict(G.out_degree())
in_deg = dict(G.in_degree())

deg_df = pd.DataFrame({"Out_Degree": pd.Series(out_deg), "In_Degree": pd.Series(in_deg)}).fillna(0)
top_regs = deg_df.sort_values("Out_Degree", ascending=False).head(10)
print("Top 10 Regulators by Out-Degree:")
print(top_regs)

In [ ]:
# PageRank
nodes_with_out = [n for n, d in G.out_degree() if d >= 1]
G_filt = G.subgraph(nodes_with_out).copy()
pr = nx.pagerank(G_filt, weight="Importance")
deg_df["PageRank"] = pd.Series(pr)

plot_df = deg_df.dropna(subset=["PageRank"])
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=plot_df, x="Out_Degree", y="PageRank",
    hue="Out_Degree", palette="viridis", size="PageRank", sizes=(20, 200)
)
top_pr = plot_df.sort_values("PageRank", ascending=False).head(5)
for gene, row in top_pr.iterrows():
    plt.text(row["Out_Degree"] + 0.5, row["PageRank"], gene, fontsize=9, weight="bold")
plt.title("Out-Degree vs. PageRank (Out-Degree >= 1)")
plt.xlabel("Out-Degree")
plt.ylabel("PageRank")
plt.show()

In [ ]:
# Betweenness centrality
for u, v, d in G.edges(data=True):
    d["distance"] = 1.0 / (d["Importance"] + 1e-5)

btw = nx.betweenness_centrality(G, weight="distance")
deg_df["Betweenness"] = pd.Series(btw)
top_btw = deg_df.sort_values("Betweenness", ascending=False)["Betweenness"].head(5)
print("Top 5 by Betweenness Centrality:")
print(top_btw)

In [ ]:
# Top 100 edges visualization
viz = thresh.head(100)
Gv = nx.from_pandas_edgelist(
    viz, source="TF", target="Target", edge_attr="Importance", create_using=nx.DiGraph()
)

source_tfs = set(viz["TF"])
node_colors = ["tomato" if n in source_tfs else "lightblue" for n in Gv.nodes()]
node_sizes = [(Gv.degree(n) * 50) + 100 for n in Gv.nodes()]
weights = [Gv[u][v]["Importance"] for u, v in Gv.edges()]
max_w = max(weights)
edge_widths = [(w / max_w) * 3 for w in weights]

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(Gv, k=0.4, seed=42)
nx.draw_networkx_nodes(Gv, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)
nx.draw_networkx_edges(Gv, pos, width=edge_widths, edge_color="gray", alpha=0.6, arrowsize=12)
nx.draw_networkx_labels(Gv, pos, font_size=8, font_weight="bold")
plt.title("Top 100 Regulatory Interactions\n(Red: TFs, Blue: Targets)")
plt.axis("off")
plt.show()